# Notebook Overview
## Import CSVs Into Databricks (PySpark)
 This notebook loads the **CSV files** (prepared in Excel) from a Databricks Volume and writes them as **Delta tables** into the `shoply_analytics.bronze` schema. 

# 1. Configuration

In [0]:
# Catalog & schema for landing storage
catalog = "shoply_analytics"
schema = "bronze"

# Base folder for input files
base_path = "/Volumes/shoply_analytics/bronze/source_systems/"

# Mapping of landing tables → cleaned CSV filenames
files = {
    "customer_demographics": "customer_demographics_cleaned.csv",
    "campaigns": "campaigns_cleaned.csv",
    "conversions": "conversions_cleaned.csv",
    "page_views": "page_views_cleaned.csv",
    "sessions": "sessions_cleaned.csv"
}

print("Catalog:", catalog)
print("Schema:", schema)
print("Base Path:", base_path)




# 2. Ingestion Function

In [0]:
def ingest_landing_csv(table_name: str, filename: str):
    """
    Ingest a prepared CSV file into a Delta table in the bronze schema.

    table_name : Name of the landing Delta table to write to.
    filename   : Name of the cleaned CSV file located in the Volume.
    """
    
    source_path = f"{base_path}{filename}"
    full_table_name = f"{catalog}.{schema}.{table_name}"

    print(f"\nReading file: {source_path}")

    # Read prepared CSV into DataFrame
    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(source_path)
    )

    print(f"Writing Delta table: {full_table_name}")

    # Write DataFrame as Delta landing table
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(full_table_name)
    )

    print(f"Completed ingestion for: {table_name}")


# 3. Loop Through All Files and Ingest

In [0]:
for table, file in files.items():
    ingest_landing_csv(table, file)


# 4. Validation

In [0]:
spark.table("shoply_analytics.bronze.sessions").limit(10).display()
